## Silver - fact_pagamento

### Transformações para a camada silver

####Importando bibliotecas

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import trim, col
from pyspark.sql.types import StringType
from datetime import datetime
import pandas as pd



### fact_pagamento
####Consultando dados da tabela bronze

In [0]:
df_pagamento = spark.table("dev_credit_risk.bronze.fact_pagamento")
df_pagamento.limit(100).display()

#### Início das transformações
#### Retirando os espaços em branco das colunas de tipo string.

In [0]:
for field in df_pagamento.schema.fields:
  if isinstance(field.dataType, StringType):
    df_pagamento = df_pagamento.withColumn(field.name, trim(col(field.name)))

df_pagamento.limit(100).display()

####Transformando colunas de data em tipo date e normalizando valores de data

In [0]:
from pyspark.sql.functions import col, try_to_date, coalesce

def corrigir_multiplas_colunas_data(df, colunas, formatos=None):
    """
    Corrige múltiplos formatos de data em várias colunas de um DataFrame
    usando try_to_date para evitar quebras de parsing no Spark moderno.
    """
    if formatos is None:
        # Coloquei o formato com barra primeiro para priorizar o seu cenário
        formatos = ["yyyy/MM/dd", "yyyy-MM-dd", "dd/MM/yyyy", "dd-MM-yyyy"]
    
    for nome_coluna in colunas:
        # Trocamos to_date por try_to_date
        tentativas = [try_to_date(col(nome_coluna), fmt) for fmt in formatos]
        df = df.withColumn(nome_coluna, coalesce(*tentativas))
        
    return df

# --- Aplicação ---
colunas_para_corrigir = ["dtVencimento", "dtPagamento", "dtCompetencia"]

df_pagamento = corrigir_multiplas_colunas_data(df_pagamento, colunas_para_corrigir)
display(df_pagamento.limit(100))

####Exibindo o schema, quantidade de linhas e summary

In [0]:
df_pagamento.printSchema()
qtd_linhas = df_pagamento.count()
print("Qtd de linhas = " + str(qtd_linhas))
display(df_pagamento.summary())

###Padronizando os nomes das colunas

In [0]:
RENAME_MAP = {
    "idPagamento": "id_pagamento",
    "contratoId": "id_contrato",
    "dtVencimento": "dt_vencimento",
    "dtPagamento": "dt_pagamento",
    "dtCompetencia": "dt_competencia",
    "valorPago": "valor_pago",
    "diasAtraso": "dias_atraso"
}

####Renomeando as colunas

In [0]:
for old_name, new_name in RENAME_MAP.items():
    df_pagamento = df_pagamento.withColumnRenamed(old_name, new_name)


####Exibindo o dataframe pronto para ser inserido na tabela delta silver

In [0]:
df_pagamento.limit(100).display()

####Inserindo os dados na tabela silver fact_pagamento

In [0]:
(df_pagamento.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("dev_credit_risk.silver.fact_pagamento")
)

####Consultando os dados com SQL no schema tabela delta silver fact_pagamento

In [0]:
%sql
select * from dev_credit_risk.silver.fact_pagamento limit 10